In [1]:
import json
import hashlib
import itertools
import numpy as np
from pathlib import Path

DATA_DIR = Path("data/prepared_smoke")
CFG_PATH = Path("configs/grid.yaml")

import yaml
cfg = yaml.safe_load(CFG_PATH.read_text())

# ---- Load everything ----
p1 = json.loads((DATA_DIR / "registry.json").read_text())
sub = json.loads((DATA_DIR / "subsample_registry.json").read_text())
seeds_cfg = cfg.get("synthetic_seeds", cfg.get("seeds", []))
vols_cfg  = cfg.get("synthetic_volume_levels",
                    cfg.get("volume_levels", []))

failures = []
def check(name, cond):
    ok = bool(cond)
    print(f"{'✓' if ok else '✗'} {name}")
    if not ok:
        failures.append(name)

print("=" * 70)
print("PHASE 1: Per-Dataset Audit")
print("=" * 70)

for key, meta in p1["datasets"].items():
    data = np.load(DATA_DIR / f"{key}.npz")
    train_idx = data["train_idx"]
    val_idx   = data["val_idx"]

    # 1. Digests match bytes
    check(f"[{key}] val_digest matches bytes",
          hashlib.sha256(val_idx.tobytes()).hexdigest() == meta["val_digest"])
    check(f"[{key}] train_digest matches bytes",
          hashlib.sha256(train_idx.tobytes()).hexdigest() == meta["train_digest"])

    # 2. Disjointness
    check(f"[{key}] train ∩ val == ∅",
          len(set(train_idx.tolist()) & set(val_idx.tolist())) == 0)

    # 3. Completeness: train + val == a single contiguous universe
    # (In Phase 1 the universe is [0, n_original); we can infer n_original
    #  from the union size, then check nothing is missing.)
    union = np.union1d(train_idx, val_idx)
    check(f"[{key}] train ∪ val is contiguous [0, n)",
          np.array_equal(union, np.arange(union.size)))

    # 4. Array shapes match index arrays
    check(f"[{key}] X_train.shape[0] == len(train_idx)",
          data["X_train"].shape[0] == len(train_idx))
    check(f"[{key}] X_val.shape[0] == len(val_idx)",
          data["X_val"].shape[0] == len(val_idx))

    # 5. Sorted (deterministic hashing)
    check(f"[{key}] train_idx sorted",
          np.all(train_idx[:-1] <= train_idx[1:]))
    check(f"[{key}] val_idx sorted",
          np.all(val_idx[:-1] <= val_idx[1:]))

print()
print("=" * 70)
print("PHASE 2: Per-Experiment Audit")
print("=" * 70)

# Group subsample entries by dataset
by_dataset = {}
for exp_key, entry in sub.items():
    by_dataset.setdefault(entry["dataset_key"], []).append((exp_key, entry))

for ds_key, entries in by_dataset.items():
    if ds_key not in p1["datasets"]:
        check(f"[{ds_key}] exists in Phase 1 registry", False)
        continue

    data = np.load(DATA_DIR / f"{ds_key}.npz")
    train_idx = data["train_idx"]
    val_idx   = set(data["val_idx"].tolist())
    n_train   = len(train_idx)

    print(f"\n--- {ds_key}: {len(entries)} experiments ---")

    for exp_key, entry in entries:
        local  = np.array(entry["local_indices"],  dtype=np.int64)
        global_ = np.array(entry["global_indices"], dtype=np.int64)

        # Digest
        check(f"[{exp_key}] digest matches",
              hashlib.sha256(local.tobytes()).hexdigest() == entry["indices_digest"])

        # Size
        check(f"[{exp_key}] n_samples == len(local) == volume_level",
              entry["n_samples"] == len(local) == entry["volume_level"])

        # Local → global mapping
        check(f"[{exp_key}] global == train_idx[local]",
              np.array_equal(global_, train_idx[local]))

        # No leakage: global training indices vs global validation indices
        check(f"[{exp_key}] no val contamination",
              len(set(global_.tolist()) & val_idx) == 0)

        # In-range local
        check(f"[{exp_key}] local in [0, n_train)",
              local.min() >= 0 and local.max() < n_train)

        # Sorted local indices
        check(f"[{exp_key}] local sorted",
              np.all(local[:-1] <= local[1:]))

        # Seed ∈ config
        check(f"[{exp_key}] seed ∈ config",
              entry["seed"] in seeds_cfg)

    # Coverage: every (int volume ≤ n_train, seed) combination present?
    int_vols = [v for v in vols_cfg if isinstance(v, int) and v <= n_train]
    if not int_vols:
        int_vols = [n_train]  # fallback resolution
    expected_keys = {
        f"{ds_key}_vol{v}_seed{s}" for v, s in itertools.product(int_vols, seeds_cfg)
    }
    present_keys = {k for k, _ in entries}
    missing = expected_keys - present_keys
    extra   = present_keys - expected_keys
    check(f"[{ds_key}] all expected experiments present", not missing)
    check(f"[{ds_key}] no unexpected experiments", not extra)
    if missing:
        print(f"    missing: {sorted(missing)}")
    if extra:
        print(f"    extra:   {sorted(extra)}")

print()
print("=" * 70)
print(f"RESULT: {len(failures)} failure(s)")
print("=" * 70)
for f in failures:
    print(f"  ✗ {f}")
if not failures:
    print("  ✓ All checks passed — pipeline is audited end-to-end.")

PHASE 1: Per-Dataset Audit
✓ [synth_shift] val_digest matches bytes
✓ [synth_shift] train_digest matches bytes
✓ [synth_shift] train ∩ val == ∅
✓ [synth_shift] train ∪ val is contiguous [0, n)
✓ [synth_shift] X_train.shape[0] == len(train_idx)
✓ [synth_shift] X_val.shape[0] == len(val_idx)
✓ [synth_shift] train_idx sorted
✓ [synth_shift] val_idx sorted
✓ [synth_iid] val_digest matches bytes
✓ [synth_iid] train_digest matches bytes
✓ [synth_iid] train ∩ val == ∅
✓ [synth_iid] train ∪ val is contiguous [0, n)
✓ [synth_iid] X_train.shape[0] == len(train_idx)
✓ [synth_iid] X_val.shape[0] == len(val_idx)
✓ [synth_iid] train_idx sorted
✓ [synth_iid] val_idx sorted

PHASE 2: Per-Experiment Audit

--- synth_shift: 20 experiments ---
✓ [synth_shift_vol10000_seed0] digest matches
✓ [synth_shift_vol10000_seed0] n_samples == len(local) == volume_level
✓ [synth_shift_vol10000_seed0] global == train_idx[local]
✓ [synth_shift_vol10000_seed0] no val contamination
✓ [synth_shift_vol10000_seed0] local i

# The Ultimate "Same Indices" Guarantee (Simulating Phase 3 & 4)

In [2]:
import json
import numpy as np

print("\n=== PHASE 3/4 SIMULATION ===")

# 1. Load the raw prepared data (Phase 1 output)
data = np.load('data/prepared_smoke/synth_shift.npz')
X_train_full = data['X_train']
y_train_full = data['y_train']

# 2. Load the subsample registry (Phase 2 output)
with open('data/prepared_smoke/subsample_registry.json') as f:
    sub_reg = json.load(f)

# 3. Both branches look up the SAME key
experiment_key = "synth_shift_vol10000_seed0"
entry = sub_reg[experiment_key]

# 4. Both branches slice the data using the EXACT SAME local indices
local_idx = entry['local_indices']
X_subset = X_train_full[local_idx]
y_subset = y_train_full[local_idx]

print(f"Experiment: {experiment_key}")
print(f"Boosting Branch (XGBoost) sees:    X shape {X_subset.shape}, y shape {y_subset.shape}")
print(f"Foundation Branch (TabPFN) sees:   X shape {X_subset.shape}, y shape {y_subset.shape}")

# 5. Prove they are identical
identical = np.array_equal(X_subset, X_train_full[local_idx])
print(f"Are the training sets identical across branches? {identical}")

# 6. Prove that the validation and test sets remain completely untouched
X_val = data['X_val']
X_id  = data['X_id']
print(f"Validation set is fixed: {X_val.shape[0] == len(data['val_idx'])}")
print(f"ID Test set is fixed:    {X_id.shape[0] == data['y_id'].shape[0]}")

# 7. Bonus: show that global_indices map back to the original dataset
train_idx_global = data['train_idx']
global_idx = entry['global_indices']
print(f"Global indices map correctly: {np.array_equal(np.array(global_idx), train_idx_global[local_idx])}")


=== PHASE 3/4 SIMULATION ===
Experiment: synth_shift_vol10000_seed0
Boosting Branch (XGBoost) sees:    X shape (10000, 8), y shape (10000,)
Foundation Branch (TabPFN) sees:   X shape (10000, 8), y shape (10000,)
Are the training sets identical across branches? True
Validation set is fixed: True
ID Test set is fixed:    True
Global indices map correctly: True
